In [23]:
# LogisticRegression
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Load the data
# ============================================================

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")


# Separate features (X) and target (y)
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]

X_val = val_df.drop("Class", axis=1)
y_val = val_df["Class"]

X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]


# ============================================================
# 2. Scale the data
# ============================================================

# Fit the scaler ONLY on the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler for validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 3. Create the individual models
# ============================================================

logistic_model = LogisticRegression()


# ============================================================
# 5. Train the Voting Classifier
# ============================================================

logistic_model.fit(X_train_scaled, y_train)


# ============================================================
# 6. Get validation probabilities
# ============================================================

# predict_proba() returns the probability of each class.
#
# For binary classification:
#
# [:, 0] -> probability of class 0
# [:, 1] -> probability of class 1

val_probabilities = logistic_model.predict_proba(X_val_scaled)[:, 1]


# ============================================================
# 7. Find the best threshold using Precision-Recall Curve
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_probabilities
)


# Calculate F1 for every threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)


# Find the index of the maximum F1
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)


# ============================================================
# 8. Evaluate the model on the TEST set
# ============================================================

# Get probabilities for the positive class
test_probabilities = logistic_model.predict_proba(X_test_scaled)[:, 1]


# Apply the threshold we found using validation data
y_test_pred = (test_probabilities >= best_threshold).astype(int)


# ============================================================
# 9. Calculate test metrics
# ============================================================

accuracy = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test = recall_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, test_probabilities)


print("\n================ TEST RESULTS ================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall   : {recall_test:.4f}")
print(f"F1 Score : {f1_test:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")


# ============================================================
# 10. Classification report
# ============================================================

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


# ============================================================
# 11. Confusion Matrix
# ============================================================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Best threshold: 0.24680181262381928
Best validation F1: 0.8143712574353329

================ TEST RESULTS ================
Accuracy : 0.9992
Precision: 0.7653
Recall   : 0.7732
F1 Score : 0.7692
ROC AUC  : 0.9799

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.77      0.77      0.77        97

    accuracy                           1.00     56960
   macro avg       0.88      0.89      0.88     56960
weighted avg       1.00      1.00      1.00     56960

Confusion Matrix:
[[56840    23]
 [   22    75]]


In [20]:
#RandomForest
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Load the data
# ============================================================

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")


# Separate features (X) and target (y)
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]

X_val = val_df.drop("Class", axis=1)
y_val = val_df["Class"]

X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]


# ============================================================
# 2. Scale the data
# ============================================================

# Fit the scaler ONLY on the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler for validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 3. Create the individual models
# ============================================================


random_forest_model = RandomForestClassifier()


# ============================================================
# 5. Train the Voting Classifier
# ============================================================

random_forest_model.fit(X_train_scaled, y_train)


# ============================================================
# 6. Get validation probabilities
# ============================================================

# predict_proba() returns the probability of each class.
#
# For binary classification:
#
# [:, 0] -> probability of class 0
# [:, 1] -> probability of class 1

val_probabilities = random_forest_model.predict_proba(X_val_scaled)[:, 1]


# ============================================================
# 7. Find the best threshold using Precision-Recall Curve
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_probabilities
)


# Calculate F1 for every threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)


# Find the index of the maximum F1
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)


# ============================================================
# 8. Evaluate the model on the TEST set
# ============================================================

# Get probabilities for the positive class
test_probabilities = random_forest_model.predict_proba(X_test_scaled)[:, 1]


# Apply the threshold we found using validation data
y_test_pred = (test_probabilities >= best_threshold).astype(int)


# ============================================================
# 9. Calculate test metrics
# ============================================================

accuracy = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test = recall_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, test_probabilities)


print("\n================ TEST RESULTS ================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall   : {recall_test:.4f}")
print(f"F1 Score : {f1_test:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")


# ============================================================
# 10. Classification report
# ============================================================

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


# ============================================================
# 11. Confusion Matrix
# ============================================================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Best threshold: 0.34
Best validation F1: 0.8837209301826663

================ TEST RESULTS ================
Accuracy : 0.9995
Precision: 0.8384
Recall   : 0.8557
F1 Score : 0.8469
ROC AUC  : 0.9519

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.84      0.86      0.85        97

    accuracy                           1.00     56960
   macro avg       0.92      0.93      0.92     56960
weighted avg       1.00      1.00      1.00     56960

Confusion Matrix:
[[56847    16]
 [   14    83]]


In [28]:
# Voting
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Load the data
# ============================================================

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/trainval.csv")
test_df = pd.read_csv("../data/test.csv")


# Separate features (X) and target (y)
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]

X_val = val_df.drop("Class", axis=1)
y_val = val_df["Class"]

X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]

# ============================================================
# 2. Scale the data
# ============================================================

# Fit the scaler ONLY on the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler for validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_val = np.concatenate([X_train_scaled, X_val_scaled], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)
# ============================================================
# 3. Create the individual models
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 4. Create the Voting Classifier
# ============================================================

voting_model = VotingClassifier(
    estimators=[
        ("lr", logistic_model),
        ("rf", random_forest_model)
    ],
    voting="soft",

    # Random Forest has twice the importance of Logistic Regression
    weights=[1, 2]
)


# ============================================================
# 5. Train the Voting Classifier
# ============================================================

voting_model.fit(X_train_scaled, y_train)


# ============================================================
# 6. Get validation probabilities
# ============================================================

# predict_proba() returns the probability of each class.
#
# For binary classification:
#
# [:, 0] -> probability of class 0
# [:, 1] -> probability of class 1

val_probabilities = voting_model.predict_proba(X_test_scaled)[:, 1]


# ============================================================
# 7. Find the best threshold using Precision-Recall Curve
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_test,
    val_probabilities
)


# Calculate F1 for every threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)


# Find the index of the maximum F1
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)


# ============================================================
# 8. Evaluate the model on the TEST set
# ============================================================

# Get probabilities for the positive class
test_probabilities = voting_model.predict_proba(X_test_scaled)[:, 1]


# Apply the threshold we found using validation data
y_test_pred = (test_probabilities >= best_threshold).astype(int)


# ============================================================
# 9. Calculate test metrics
# ============================================================

accuracy = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test = recall_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, test_probabilities)


print("\n================ TEST RESULTS ================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall   : {recall_test:.4f}")
print(f"F1 Score : {f1_test:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")


# ============================================================
# 10. Classification report
# ============================================================

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


# ============================================================
# 11. Confusion Matrix
# ============================================================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Best threshold: 0.4459729699561434
Best validation F1: 0.851063829737285

================ TEST RESULTS ================
Accuracy : 0.9995
Precision: 0.8791
Recall   : 0.8247
F1 Score : 0.8511
ROC AUC  : 0.9782

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.88      0.82      0.85        97

    accuracy                           1.00     56960
   macro avg       0.94      0.91      0.93     56960
weighted avg       1.00      1.00      1.00     56960

Confusion Matrix:
[[56852    11]
 [   17    80]]


In [30]:
# Voting 4 Models
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    VotingClassifier
)
from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Load the data
# ============================================================

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")


# Separate features (X) and target (y)
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]

X_val = val_df.drop("Class", axis=1)
y_val = val_df["Class"]

X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]


# ============================================================
# 2. Scale the data
# ============================================================

# Fit the scaler ONLY on the training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler for validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 3. Create the individual models
# ============================================================


random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)

# probability=True is REQUIRED here because we are using soft voting
svc_model = SVC(
    probability=True,
    random_state=42
)

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)


# ============================================================
# 4. Create the Voting Classifier
# ============================================================

voting_model = VotingClassifier(
    estimators=[
        ("rf", random_forest_model),
        ("gb", gradient_boosting_model),
        ("svc", svc_model),
        ("knn", knn_model)
    ],
    voting="soft",

    # All 4 models are given equal weighting
    weights=[1, 1, 1, 1]
)


# ============================================================
# 5. Train the Voting Classifier
# ============================================================

voting_model.fit(X_train_scaled, y_train)


# ============================================================
# 6. Get validation probabilities
# ============================================================

# predict_proba() returns the probability of each class.
#
# For binary classification:
#
# [:, 0] -> probability of class 0
# [:, 1] -> probability of class 1

val_probabilities = voting_model.predict_proba(X_val_scaled)[:, 1]


# ============================================================
# 7. Find the best threshold using Precision-Recall Curve
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_probabilities
)


# Calculate F1 for every threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)


# Find the index of the maximum F1
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)


# ============================================================
# 8. Evaluate the model on the TEST set
# ============================================================

# Get probabilities for the positive class
test_probabilities = voting_model.predict_proba(X_test_scaled)[:, 1]


# Apply the threshold we found using validation data
y_test_pred = (test_probabilities >= best_threshold).astype(int)


# ============================================================
# 9. Calculate test metrics
# ============================================================

accuracy = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test = recall_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, test_probabilities)


print("\n================ TEST RESULTS ================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall   : {recall_test:.4f}")
print(f"F1 Score : {f1_test:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")


# ============================================================
# 10. Classification report
# ============================================================

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))


# ============================================================
# 11. Confusion Matrix
# ============================================================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

/Users/macbook/Desktop/NewML copy 2/.ml/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best threshold: 0.24518712410003998
Best validation F1: 0.8875739644472533

================ TEST RESULTS ================
Accuracy : 0.9995
Precision: 0.8485
Recall   : 0.8660
F1 Score : 0.8571
ROC AUC  : 0.9708

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.85      0.87      0.86        97

    accuracy                           1.00     56960
   macro avg       0.92      0.93      0.93     56960
weighted avg       1.00      1.00      1.00     56960

Confusion Matrix:
[[56848    15]
 [   13    84]]
